# Figure 3j,k — Skeleton traces with synapse locations

Each sensory axon is drawn as a flat (x–y) skeleton trace with its synapses overlaid as
dots, one figure per neuron, for the **16 manually validated axons** (2 per subtype).

| Panel | Dots | Synapse direction |
| --- | --- | --- |
| **j** | axodendritic synapses (blue) | sensory neuron **presynaptic** |
| **k** | axoaxonic synapses | sensory neuron **postsynaptic** |

Skeletons come from `../data/sensory_neuron_swc/` (SWC exports, shipped with the repo);
synapse positions are queried live from the CAVE `sensory_neuron_synapses` table, so this
notebook **needs a CAVE token**. Rebuild the SWCs with
`data_generation/export_sensory_neuron_swc.ipynb`.

Both sets of axes are placed in **pixel** coordinates and aligned to the soma, so traces
from different neurons are directly comparable. Panel j additionally runs a two-pass
version that computes common x/y limits across all 16 neurons first, so every published
trace shares one scale. Note the synapse table and the SWC files use different voxel
resolutions (16×16×45 nm and 32×32×45 nm), which the plotting functions convert between —
don't change one without the other.


In [ ]:
# Libraries
import caveclient
import subprocess
import json
import numpy as np
import matplotlib.pyplot as plt
# Setup caveclient
client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr", ) #server_address='https://global.daf-apis.com')

In [ ]:
sensory_tag = client.materialize.query_table('sensory_neuron_label')

In [ ]:
# --- helpers required by plot_swc_xy_in_px_soma_fixed (no plotting changes) ---
from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection  # plotting uses this

def load_swc_df(path: str) -> pd.DataFrame:
    return pd.read_csv(
        path, comment="#", delim_whitespace=True, header=None,
        names=["n","type","x","y","z","radius","parent"],
        dtype={"n": int, "type": int, "x": float, "y": float, "z": float, "radius": float, "parent": int},
    )

def _detect_units_from_header(path: str) -> str:
    # default to nm if unspecified
    units = "nm"
    with open(path, "r") as f:
        for line in f:
            if not line.startswith("#"):
                break
            if "Units:" in line:
                u = line.split("Units:")[-1].strip().lower().split()[0]
                if u in ("nm", "um", "px"):
                    units = u
                break
    return units

def _to_pixels(coords: np.ndarray, file_units: str, res_nm_per_px=(32.0, 32.0, 45.0)) -> np.ndarray:
    """
    Convert SWC coordinates to voxel/pixel units used for plotting.
    - If SWC units == 'px': return as-is.
    - If 'nm': divide by res_nm_per_px.
    - If 'um': convert to nm then divide by res_nm_per_px.
    """
    coords = np.asarray(coords, float)
    res = np.asarray(res_nm_per_px, float)[None, :]
    if file_units == "px":
        return coords
    elif file_units == "nm":
        return coords / res
    elif file_units == "um":
        return (coords * 1000.0) / res
    else:
        # fallback: assume nm
        return coords / res

def _infer_soma_px_from_swc(df: pd.DataFrame, file_units: str, res_nm_per_px=(32.0, 32.0, 45.0)):
    """
    Use the root node (parent == -1) as soma proxy, converted to px.
    If multiple roots, take the first; if none, use row 0.
    """
    if "parent" in df.columns:
        root_idx = int(np.where(df["parent"].to_numpy(int) == -1)[0][0]) if (df["parent"] == -1).any() else 0
    else:
        root_idx = 0
    c = df.loc[root_idx, ["x","y","z"]].to_numpy(float).reshape(1, 3)
    c_px = _to_pixels(c, file_units, res_nm_per_px=res_nm_per_px)
    return c_px.ravel().tolist()

def _cm_to_inches(w_cm: float, h_cm: float):
    return (w_cm / 2.54, h_cm / 2.54)


## Position helpers

The original working notebook called `_parse_position` and `synapse_xy_nm_for_plot`
without defining them — they lived in the interactive session and were never saved
(its own comment says "assumes these are already defined in your session"). They are
restored here from the equivalent functions in `plot_synapse_with_swc_skeleton.ipynb`,
so this notebook is self-contained.

`_parse_position` also accepts the whitespace-separated form CAVE writes to CSV
(`"[7581. 5484. 1324.]"`), which `ast.literal_eval` alone cannot parse.


In [ ]:
import ast

import numpy as np


def _parse_position(val):
    """Return an (x, y, z) float array from a list/tuple/array or a string.

    Handles both "[x, y, z]" and the whitespace-separated "[x y z]" that CAVE
    positions take once they have been through a CSV round-trip.
    """
    if isinstance(val, (list, tuple, np.ndarray)):
        arr = np.asarray(val, dtype=float).ravel()
    elif isinstance(val, str):
        try:
            arr = np.asarray(ast.literal_eval(val), dtype=float).ravel()
        except (ValueError, SyntaxError):
            arr = np.fromstring(val.strip("[]() "), sep=" ", dtype=float)
    else:
        raise ValueError(f"unsupported position type: {type(val)}")
    if arr.size != 3:
        raise ValueError(f"position must have 3 numbers, got {arr}")
    return arr


def synapse_xy_nm_for_plot(df, *, pos_col="ctr_pt_position", pos_units="px",
                           res_nm_per_px=(16.0, 16.0, 45.0)) -> np.ndarray:
    """(N, 2) synapse x/y in nanometres, for overlaying on a skeleton trace."""
    if df is None or len(df) == 0:
        return np.empty((0, 2), float)
    sel = df.loc[df[pos_col].notna()]
    if sel.empty:
        return np.empty((0, 2), float)

    pos = np.vstack(sel[pos_col].apply(_parse_position).tolist()).astype(float)

    if pos_units == "px":
        pos_nm = pos * np.asarray(res_nm_per_px, float)[None, :]
    elif pos_units == "um":
        pos_nm = pos * 1000.0
    else:                      # "nm", or unknown -> assume already nm
        pos_nm = pos
    return pos_nm[:, :2]


In [ ]:
def plot_swc_xy_in_px_soma_fixed(
    swc_path,
    soma_px=None,                      # [x_px, y_px, z_px]; if None, will infer from SWC
    *,
    res_nm_per_px=(32.0, 32.0, 45.0),  # divide skeleton coords by this
    linewidth=0.6,
    alpha_dend=0.7,
    alpha_axon=0.35,
    alpha_soma=1.0,
    soma_size=900,
    soma_edgewidth=1.6,
    figsize_cm=(8, 8),
    equal_aspect=True,
    invert_y=True,
    show_axon=True,
    axon_color="r",
    dend_color="k",
    minimalist=True,
    save_svg=False,
    svg_path=None,
    print_soma_nm=True,
    debug=False,
    # presynaptic synapses (nm)
    syn_xy_nm=None,
    syn_size=12.0,
    syn_color="tab:blue",
    syn_alpha=0.8,
    syn_edgecolors="none",
    # input synapses (nm)  <<< NEW
    in_syn_xy_nm=None,
    in_syn_size=12.0,
    in_syn_color="tab:green",
    in_syn_alpha=0.8,
    in_syn_edgecolors="none",
):
    # Load SWC + convert skeleton coords to px
    df = load_swc_df(swc_path)
    file_units = _detect_units_from_header(swc_path)
    coords_px = _to_pixels(df[["x","y","z"]].to_numpy(), file_units, res_nm_per_px)

    # If soma not provided, infer from SWC
    if soma_px is None:
        soma_px = _infer_soma_px_from_swc(df, file_units, res_nm_per_px)

    soma_px = np.asarray(soma_px, float).ravel()
    if soma_px.size != 3:
        raise ValueError(f"soma_px must be 3 numbers [x,y,z], got {soma_px.tolist()}")

    # Your original style: compute soma_nm and plot with it
    soma_nm = soma_px * np.asarray(res_nm_per_px, float)
    if print_soma_nm:
        print(f"Soma (nm): x={soma_nm[0]:.3f}, y={soma_nm[1]:.3f}, z={soma_nm[2]:.3f}")

    if debug:
        print(f"[Units] SWC file_units={file_units}")
        print(f"[Resolution nm/px] {res_nm_per_px}")
        print("First 3 SWC nodes in px (post-division):")
        print(coords_px[:3])

    # Build XY segments in px
    types   = df["type"].to_numpy(int)
    parents = df["parent"].to_numpy(int)
    ids     = df["n"].to_numpy(int)
    id_to_idx = {int(n): i for i, n in enumerate(ids)}

    axon_segs, dend_segs = [], []
    for i, p in enumerate(parents):
        if p == -1: 
            continue
        j = id_to_idx.get(int(p))
        if j is None:
            continue
        xi, yi = coords_px[i, 0], coords_px[i, 1]
        xj, yj = coords_px[j, 0], coords_px[j, 1]
        if types[i] == 2 and types[j] == 2:
            axon_segs.append([(xj, yj), (xi, yi)])
        else:
            dend_segs.append([(xj, yj), (xi, yi)])

    # Figure in centimeters
    fig, ax = plt.subplots(figsize=_cm_to_inches(*figsize_cm))

    # Layering: axon (back) < dendrite < syn (presyn) < in_syn (inputs) < soma
    if show_axon and axon_segs:
        ax.add_collection(LineCollection(axon_segs, colors=axon_color, linewidths=linewidth, alpha=alpha_axon, zorder=1))
    if dend_segs:
        ax.add_collection(LineCollection(dend_segs, colors=dend_color, linewidths=linewidth, alpha=alpha_dend, zorder=4))

    # Presynaptic synapses (nm)
    if syn_xy_nm is not None:
        syn_xy_nm = np.asarray(syn_xy_nm, float)
        if syn_xy_nm.ndim == 1 and syn_xy_nm.size == 0:
            syn_xy_nm = syn_xy_nm.reshape(0,2)
        if syn_xy_nm.size:
            ax.scatter(syn_xy_nm[:,0], syn_xy_nm[:,1],
                       s=syn_size, c=syn_color, alpha=syn_alpha,
                       edgecolors=syn_edgecolors, zorder=6)

    # Input synapses (nm)  <<< NEW
    if in_syn_xy_nm is not None:
        in_syn_xy_nm = np.asarray(in_syn_xy_nm, float)
        if in_syn_xy_nm.ndim == 1 and in_syn_xy_nm.size == 0:
            in_syn_xy_nm = in_syn_xy_nm.reshape(0,2)
        if in_syn_xy_nm.size:
            ax.scatter(in_syn_xy_nm[:,0], in_syn_xy_nm[:,1],
                       s=in_syn_size, c=in_syn_color, alpha=in_syn_alpha,
                       edgecolors=in_syn_edgecolors, zorder=7)

    # Soma on top (using soma_nm per your approach)
    ax.scatter([soma_nm[0]], [soma_nm[1]],
               s=soma_size, facecolors="0.7", edgecolors="k",
               linewidths=0.2, alpha=alpha_soma, zorder=10)

    # Limits include skeleton (px) + soma (px)
    x, y = coords_px[:, 0], coords_px[:, 1]
    x_all = np.concatenate([x, [soma_px[0]]])
    y_all = np.concatenate([y, [soma_px[1]]])
    pad = 0.02 * max(x_all.max() - x_all.min(), y_all.max() - y_all.min(), 1.0)
    ax.set_xlim(x_all.min() - pad, x_all.max() + pad)
    ax.set_ylim(y_all.min() - pad, y_all.max() + pad)

    if equal_aspect:
        ax.set_aspect("equal", adjustable="box")
    if invert_y:
        ax.invert_yaxis()

    if minimalist:
        ax.set_axis_off()
    else:
        ax.set_xlabel("X (px)")
        ax.set_ylabel("Y (px)")
        ax.set_title(Path(swc_path).name)

    plt.tight_layout(pad=0)

    if save_svg:
        out = Path(svg_path) if svg_path is not None else Path(swc_path).with_suffix(".svg")
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, bbox_inches="tight", transparent=True)
        print(f"[saved] {out}")

    return fig, ax


## Panel k — axoaxonic synapses

In [ ]:
import os
from pathlib import Path
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import caveclient

# Assumes your helpers already exist in scope:
# _parse_position, synapse_xy_nm_for_plot, plot_swc_xy_in_px_soma_fixed,
# load_swc_df, _detect_units_from_header, _to_pixels, _infer_soma_px_from_swc

def batch_plot_inputs_svg_px_strict(
    client,
    root_ids,
    sensory_tag: pd.DataFrame,
    *,
    swc_dir: str,
    out_dir: str,
    syn_table: str = "sensory_neuron_synapses",
    syn_pos_col: str = "ctr_pt_position",
    syn_pos_units: str = "px",
    res_nm_per_px_syn=(16.0, 16.0, 45.0),
    res_nm_per_px_swc=(32.0, 32.0, 45.0),
):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    idx = sensory_tag.set_index("pt_root_id", drop=False)

    for k, rid in enumerate(root_ids, 1):
        # inputs for this neuron
        df_syn = client.materialize.query_table(
            syn_table,
            filter_equal_dict={"post_pt_root_id": int(rid)}
        )
        in_syn_xy_nm = synapse_xy_nm_for_plot(
            df_syn,
            pos_col=syn_pos_col,
            pos_units=syn_pos_units,
            res_nm_per_px=res_nm_per_px_syn,
        )

        # tag + soma_px (unchanged behavior)
        tag = None
        soma_px = None
        if rid in idx.index:
            row = idx.loc[rid]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
            tag = row.get("tag", None)
            ptpos = row.get("pt_position", None)
            if ptpos is not None:
                soma_px = _parse_position(ptpos)

        # swc path
        swc_path = os.path.join(swc_dir, f"{tag}_{rid}.swc") if tag is not None else None
        if swc_path is None or not os.path.exists(swc_path):
            matches = sorted(glob(os.path.join(swc_dir, f"*_{rid}.swc")))
            swc_path = matches[0] if matches else os.path.join(swc_dir, f"{rid}.swc")

        # svg path
        base = f"{tag}_{rid}" if tag is not None else f"{rid}"
        svg_path = os.path.join(out_dir, f"{base}.svg")

        try:
            # EXACT visual params requested:
            fig, ax = plot_swc_xy_in_px_soma_fixed(
                 swc_path=swc_path,
                  soma_px=soma_px,                  # unchanged; will infer if None
                 res_nm_per_px=res_nm_per_px_swc,
                 linewidth=0.4,
            alpha_dend=0.7,
     alpha_axon=0.35,
    alpha_soma=0.0,                   # <<< hide soma
    soma_size=0,                      # <<< hide soma
           figsize_cm=(6,6),
    show_axon=True,
    axon_color="#111314",
    dend_color="#f37420",
    minimalist=True,
    save_svg=True,
    svg_path=svg_path,
    print_soma_nm=False,
    # inputs in nm (unchanged)
    in_syn_xy_nm=in_syn_xy_nm,
    in_syn_color="#005b8d",
    in_syn_size=0.5,
    in_syn_alpha=0.3,
             )

            plt.close(fig)
            print(f"[{k:02d}/{len(root_ids)}] saved: {svg_path}  (n_syn={len(df_syn)})")
        except Exception as e:
            print(f"[{k:02d}/{len(root_ids)}] ERROR for {rid}: {e}")

# ---- Call it on your list ----
root_ids = [
    720575940891163574, 720575940922958792,
    720575940901960430, 720575940920833705,
    720575940893392149, 720575940902363320,
    720575940881926778, 720575940932756628,
    720575940883866908, 720575940882155492,
    720575940847552365, 720575940897347842,
    720575940918288693, 720575940881918801,
    720575940883734520, 720575940927175960,
]

swc_dir = "../data/sensory_neuron_swc"
out_dir = "figure3j_axodendritic_svg"

client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr")

batch_plot_inputs_svg_px_strict(
    client,
    root_ids,
    sensory_tag,  # must have: pt_root_id, tag, pt_position
    swc_dir=swc_dir,
    out_dir=out_dir,
    syn_table="sensory_neuron_synapses",
    syn_pos_col="ctr_pt_position",
    syn_pos_units="px",
    res_nm_per_px_syn=(16.0, 16.0, 45.0),
    res_nm_per_px_swc=(32.0, 32.0, 45.0),
)


In [ ]:
# The 16 manually validated axons, 2 per subtype.
root_ids = [
    720575940891163574, 720575940922958792,   # C-LTMR
    720575940901960430, 720575940920833705,   # C-HTMR/Heat (MrgprD/A3/B4)
    720575940893392149, 720575940902363320,   # Aδ-LTMR
    720575940881926778, 720575940932756628,   # Aβ-LTMR
    720575940883866908, 720575940882155492,   # C-HTMR/Heat (SST)
    720575940847552365, 720575940897347842,   # Aδ-HTMR
    720575940918288693, 720575940881918801,   # peptidergic C-fibre
    720575940883734520, 720575940927175960,   # C-Cold
]

SWC_DIR = "../data/sensory_neuron_swc"

batch_plot_inputs_svg_px_strict(
    client,
    root_ids,
    sensory_tag,
    swc_dir=SWC_DIR,
    out_dir="figure3k_axoaxonic_svg",
    syn_table="sensory_neuron_synapses",
    syn_pos_col="ctr_pt_position",
    syn_pos_units="px",
    res_nm_per_px_syn=(16.0, 16.0, 45.0),
    res_nm_per_px_swc=(32.0, 32.0, 45.0),
)


## Panel j — axodendritic synapses, shared axis limits

In [ ]:
import os
from pathlib import Path
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import caveclient

# Assumes these are already defined in your session (unchanged):
# _parse_position, synapse_xy_nm_for_plot, plot_swc_xy_in_px_soma_fixed,
# load_swc_df, _detect_units_from_header, _to_pixels, _cm_to_inches, _infer_soma_px_from_swc

def _find_swc_path_for(root_id, tag, swc_dir):
    if tag is not None:
        p = os.path.join(swc_dir, f"{tag}_{root_id}.swc")
        if os.path.exists(p):
            return p
    matches = sorted(glob(os.path.join(swc_dir, f"*_{root_id}.swc")))
    if matches:
        return matches[0]
    # last-chance fallback
    p = os.path.join(swc_dir, f"{root_id}.swc")
    return p if os.path.exists(p) else None

def batch_plot_inputs_svg_px_aligned(
    client,
    root_ids,
    sensory_tag: pd.DataFrame,
    *,
    swc_dir: str,
    out_dir: str,
    syn_table: str = "sensory_neuron_synapses",
    syn_pos_col: str = "ctr_pt_position",
    syn_pos_units: str = "px",
    res_nm_per_px_syn=(16.0, 16.0, 45.0),
    res_nm_per_px_swc=(32.0, 32.0, 45.0),
    common_pad_px: float = 0.0,   # set >0 if you want the same padding for all
):
    """
    Two-pass batch:
      1) Compute global X/Y limits in *pixels* across all SWCs.
      2) Plot each figure with your function (save_svg=False), apply the same limits,
         then save SVG (so all share the same top-left & bottom-right coords).
    """
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    idx = sensory_tag.set_index("pt_root_id", drop=False)

    # ---------- Pass 1: global limits in PX ----------
    xmin = np.inf; xmax = -np.inf; ymin = np.inf; ymax = -np.inf
    swc_info = []  # cache (rid, tag, soma_px, swc_path, coords_px) for pass 2

    for rid in root_ids:
        # tag + soma_px (optional)
        tag = None; soma_px = None
        if rid in idx.index:
            row = idx.loc[rid]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
            tag = row.get("tag", None)
            ptpos = row.get("pt_position", None)
            if ptpos is not None:
                soma_px = _parse_position(ptpos)

        swc_path = _find_swc_path_for(rid, tag, swc_dir)
        if swc_path is None:
            print(f"[warn] SWC not found for {rid}; skipping for limits.")
            continue

        try:
            df = load_swc_df(swc_path)
            units = _detect_units_from_header(swc_path)  # 'px' | 'nm' | 'um'
            coords_px = _to_pixels(df[["x","y","z"]].to_numpy(float), units, res_nm_per_px=res_nm_per_px_swc)
            x = coords_px[:, 0]; y = coords_px[:, 1]
            if x.size and y.size:
                xmin = min(xmin, float(np.nanmin(x)))
                xmax = max(xmax, float(np.nanmax(x)))
                ymin = min(ymin, float(np.nanmin(y)))
                ymax = max(ymax, float(np.nanmax(y)))
            swc_info.append((rid, tag, soma_px, swc_path))
        except Exception as e:
            print(f"[warn] Could not read/convert SWC for {rid}: {e}")

    if not np.isfinite([xmin, xmax, ymin, ymax]).all():
        raise RuntimeError("No valid SWCs found to compute global limits.")

    # optional uniform padding
    xmin_g = xmin - common_pad_px
    xmax_g = xmax + common_pad_px
    ymin_g = ymin - common_pad_px
    ymax_g = ymax + common_pad_px

    # ---------- Pass 2: make plots, set SAME limits, save ----------
    for k, (rid, tag, soma_px, swc_path) in enumerate(swc_info, 1):
        # query inputs and convert to nm (unchanged)
        df_syn = client.materialize.query_table(
            syn_table,
            filter_equal_dict={"pre_pt_root_id": int(rid)}
        )
        in_syn_xy_nm = synapse_xy_nm_for_plot(
            df_syn,
            pos_col=syn_pos_col,
            pos_units=syn_pos_units,
            res_nm_per_px=res_nm_per_px_syn,
        )

        base = f"{tag}_{rid}" if tag is not None else f"{rid}"
        svg_path = os.path.join(out_dir, f"{base}.svg")

        try:
            # call your plotting function exactly, BUT do not save inside it
            fig, ax = plot_swc_xy_in_px_soma_fixed(
                swc_path=swc_path,
                soma_px=soma_px,                  # will infer if None
                res_nm_per_px=res_nm_per_px_swc,
                linewidth=0.4,
                alpha_dend=0.7,
                alpha_axon=0.35,
                alpha_soma=0.0,                   # hide soma
                soma_size=0,                      # hide soma
                figsize_cm=(6,6),
                show_axon=True,
                axon_color="#111314",
                dend_color="#f37420",
                minimalist=True,
                save_svg=False,                   # we will save after setting global limits
                svg_path=None,                    # ignored
                print_soma_nm=False,
                # unchanged inputs
                in_syn_xy_nm=in_syn_xy_nm,
                in_syn_color="#005b8d",
                in_syn_size=0.5,
                in_syn_alpha=0.3,
            )

            # enforce SAME global limits (px space); keep inverted Y
            ax.set_xlim(xmin_g, xmax_g)
            ax.set_ylim(ymin_g, ymax_g)
            ax.invert_yaxis()  # ensure top-left alignment

            fig.savefig(svg_path, bbox_inches="tight", transparent=True)
            plt.close(fig)
            print(f"[{k:02d}/{len(swc_info)}] saved aligned: {svg_path}  (n_syn={len(df_syn)})")

        except Exception as e:
            print(f"[{k:02d}/{len(swc_info)}] ERROR for {rid}: {e}")


In [ ]:
root_ids = [
    720575940891163574, 720575940922958792,
    720575940901960430, 720575940920833705,
    720575940893392149, 720575940902363320,
    720575940881926778, 720575940932756628,
    720575940883866908, 720575940882155492,
    720575940847552365, 720575940897347842,
    720575940918288693, 720575940881918801,
    720575940883734520, 720575940927175960,
]

swc_dir = "../data/sensory_neuron_swc"
out_dir = "figure3j_axodendritic_svg"

client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr")

batch_plot_inputs_svg_px_aligned(
    client,
    root_ids,
    sensory_tag,                       # has pt_root_id, tag, pt_position
    swc_dir=swc_dir,
    out_dir=out_dir,
    syn_table="sensory_neuron_synapses",
    syn_pos_col="ctr_pt_position",
    syn_pos_units="px",
    res_nm_per_px_syn=(16.0, 16.0, 45.0),
    res_nm_per_px_swc=(32.0, 32.0, 45.0),
    common_pad_px=0.0,                 # set a small value (e.g., 25) if you want shared padding
)
